<a href="https://colab.research.google.com/github/TheOohWee/CSE-10124-LAB-4/blob/main/Amir_CSE10124_lab4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CSE 10124 - Building ChatGPT: Lab 04 (10 pts.)

- NETID:

## Overview

In Lab 03 you built the inner workings of a single **Transformer block** — Layer Normalization, Scaled Dot-Product Self-Attention, the Feed-Forward Network, and residual connections. In this lab we zoom out one more level: we wire the full **GPT language model** around a stack of those blocks, then train it on Shakespeare and watch the loss go down.

By the end of this lab you will have a working, from-scratch implementation of a GPT-style decoder-only transformer and a training loop that actually teaches it to predict Shakespeare one token at a time.

### What We're Building

A GPT language model takes in a sequence of token IDs $X$ of shape $(B, T)$ and outputs a probability distribution over the vocabulary for every position, of shape $(B, T, V)$. Internally, the data flows through five groups of sub-modules in series:

1. **Token Embedding** — a lookup table that turns each token ID into a $d_\text{model}$-dimensional vector
2. **Positional Embedding** — a lookup table indexed by position that tells the model *where* each token is in the sequence; we add it to the token embedding
3. **Transformer Blocks** — a stack of $N$ identical blocks (the ones you built in Lab 03), each refining the representation by mixing information across tokens
4. **Final Layer Norm** — one last normalization before the prediction head, for training stability
5. **Language Model Head** — a linear projection from $d_\text{model}$ to $V$ that produces one logit per vocabulary word; a final softmax turns logits into probabilities

You will also implement the **backward pass** that propagates the cross-entropy gradient back through this stack, and a **training loop** that uses it to reduce the loss.

### The Data Flow

```
X  (B, T)  int64  token IDs
  │
  ├─ token_embedding      ──►  tok  (B, T, C)
  ├─ positional_embedding ──►  pos  (B, T, C)
  │
  h = tok + pos
  │
  for block in transformer_blocks:
      h = block(h, key_pad_mask)          (B, T, C)
  │
  h = final_ln(h)                          (B, T, C)
  logits = lm_head(h)                      (B, T, V)
  probs  = softmax(logits, dim=-1)         (B, T, V)
```

### Topics Covered

| Task ID | Description | Points |
|---------|-------------|--------|
| 00 | Data + Library Loading (pre-built) | 0 |
| 01 | Lay Out the Architecture | 3 |
| &nbsp;&nbsp;&nbsp;&nbsp;01-1 | &nbsp;&nbsp;&nbsp;&nbsp;- Instantiate Sub-Modules | |
| &nbsp;&nbsp;&nbsp;&nbsp;01-2 | &nbsp;&nbsp;&nbsp;&nbsp;- Parameter Count Helper | |
| 02 | Write `forward` | 2 |
| &nbsp;&nbsp;&nbsp;&nbsp;02-1 | &nbsp;&nbsp;&nbsp;&nbsp;- `forward(X)` Function | |
| &nbsp;&nbsp;&nbsp;&nbsp;02-2 | &nbsp;&nbsp;&nbsp;&nbsp;- Shape + Probability Sanity Check | |
| 03 | Write `backward` | 2 |
| &nbsp;&nbsp;&nbsp;&nbsp;03-1 | &nbsp;&nbsp;&nbsp;&nbsp;- `backward(Y_hat, Y)` Function | |
| &nbsp;&nbsp;&nbsp;&nbsp;03-2 | &nbsp;&nbsp;&nbsp;&nbsp;- Gradient Non-Zero Check | |
| 04 | Training Loop | 3 |
| &nbsp;&nbsp;&nbsp;&nbsp;04-1 | &nbsp;&nbsp;&nbsp;&nbsp;- Masked `cross_entropy` | |
| &nbsp;&nbsp;&nbsp;&nbsp;04-2 | &nbsp;&nbsp;&nbsp;&nbsp;- `step` Function | |
| &nbsp;&nbsp;&nbsp;&nbsp;04-3 | &nbsp;&nbsp;&nbsp;&nbsp;- Run the Loop | |
| 05 | Generate Submission | 0 |

Please complete all sections. The tokenizer, dataset, and DataLoader are pre-built for you in Task 00 so you can stay focused on the architecture, the forward and backward passes, and the training loop.

In the code blocks, look for comments like:
```python
# TODO: some task description
```
as these are what you will be expected to fill in. For code sections longer than a line, there's an additional comment:
```python
# LINES: N
```
where `N` is the number of lines I have in my solution. This is NOT graded on, it is just a reference for you — if you have way more or way fewer it may be worth rethinking your code, but +/- 2-3 lines is nothing to worry about usually.


## Key Terms

A few definitions you'll want before diving in. Refer back here whenever a piece of jargon shows up:

- ▸ **Token** — one unit of text the model reads or predicts. For our BPE tokenizer, a token is usually a sub-word like `the` or `tion`.
- ▸ **Logits** — the raw, unnormalized scores the model produces for every word in the vocabulary at every position. Shape `(B, T, V)`. They become probabilities only after softmax.
- ▸ **Softmax** — the function that turns a vector of logits into a probability distribution that sums to 1. `softmax(x)_i = exp(x_i) / sum_j(exp(x_j))`.
- ▸ **Cross-Entropy Loss** — measures how wrong a probability prediction is. For a single position with one-hot target, it simplifies to `-log(probability assigned to the correct token)`. Lower is better; 0 is perfect confidence on the right answer.
- ▸ **Autoregressive** — generating one token at a time, where each new token is conditioned on every prior token. The way GPT-style models produce text.
- ▸ **Causal Mask** — an attention mask that prevents position `i` from looking at any position `j > i`. This is what enforces autoregressive prediction during training.
- ▸ **LM Head** (a.k.a. **language-model head**) — the final linear layer that projects from `d_model` (the hidden size) to the vocabulary size. Produces one logit per vocabulary word per position.
- ▸ **Final Layer Norm** (`ln_f` in GPT-2 terminology) — one last `LayerNorm` applied right before the LM head. Stabilizes training by keeping the residual stream's magnitude in check.
- ▸ **Padding / PAD_IDX** — when a batch contains sequences of different lengths, we pad the short ones with a dummy token so every sequence in the batch has the same length. We mask these positions out so they don't contribute to attention or loss.
- ▸ **Backpropagation** — the algorithm for computing the gradient of the loss with respect to every parameter, by propagating gradients backward through the network in reverse order of the forward pass.

## Task 00: Setup (0 pts.)
#### Loading the repo, the tokenizer, the dataset, and the DataLoader

Run the cell below to download the course repository and set up everything you need to train:

- The `irishGPT` sub-module classes you wrote in earlier labs (`EmbeddingLayer`, `PositionalEmbedding`, `Transformer`, `LayerNorm`, `LinearLayer`).
- The `Regex_Tokenizer` trained on Shakespeare (loaded from `Datasets/shakespeare_vocab.json`).
- An `IrishChatDataset` that pairs each line of Shakespeare with its one-token-shifted target (so `Y[i] = X[i+1]`), truncated to the first 2000 lines so training takes ~30 seconds instead of ~20 minutes.
- A PyTorch `DataLoader` with batch size 16 that handles padding, one-hot encoding of targets, and moving batches to your device.

You do not need to modify this cell — every later task will reference the module-level variables defined here (`VOCAB_SIZE`, `PAD_IDX`, `CTX_LEN`, `D_MODEL`, `N_LAYERS`, `N_HEADS`, `DEVICE`, `train_loader`).

### Task 00: Code (0 pts.)


In [2]:
import os

try:
    import google.colab
    REPO_URL = "https://github.com/wtheisen/nd-cse-10124-lectures.git"

    REPO_NAME = "/content/nd-cse-10124-lectures"
    L_PATH = "nd-cse-10124-lectures"

    %cd /content/
    !rm -rf {REPO_NAME}

    if not os.path.exists(REPO_NAME):
        !git clone {REPO_URL}
        %cd {L_PATH}
        !pwd

except ImportError:
    print("Unable to download repo, either:")
    print("\tA.) You're not on colab")
    print("\tB.) It has already been cloned")

import math
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

# --- Sub-module building blocks (the pieces you wrote in earlier labs) ---
#   EmbeddingLayer:      token-ID -> d_model vector lookup table
#   PositionalEmbedding: position-index -> d_model vector lookup table
#   Transformer:         one full Pre-LN transformer block (from Lab 03)
#   LayerNorm:           layer normalization (from Lab 03)
#   LinearLayer:         linear transformation X @ W.T + b (from Lab 02)
from irishGPT.embedding import EmbeddingLayer
from irishGPT.positional_embedding import PositionalEmbedding
from irishGPT.transformer import Transformer
from irishGPT.layer_norm import LayerNorm
from irishGPT.linear_layer import LinearLayer

# --- Tokenizer + dataset wrapper (already written for you) ---
from irishGPT.tokenizer import Regex_Tokenizer
from irishGPT.dataset import IrishChatDataset

# --- Pre-built: the BPE tokenizer trained on Shakespeare ---
# This is the same tokenizer you trained in earlier assignments; we just
# load its merges + vocab from disk instead of re-training it every time.
tokenizer = Regex_Tokenizer()
tokenizer.load("Datasets/shakespeare_vocab.json")
VOCAB_SIZE = len(tokenizer.vocab)
PAD_IDX    = tokenizer.special_tokens["<|pad|>"]

# --- Pre-built: the dataset + DataLoader ---
# IrishChatDataset reads one Shakespeare line at a time, wraps it in
# <|sos|> / <|eos|>, tokenizes it, and stores (X, Y) pairs where
# Y = X shifted by one token (the next-token prediction target).
# We truncate to the first 2000 lines to keep training fast.
dataset = IrishChatDataset("Datasets/shakespeare.txt", tokenizer)
dataset.data = dataset.data[:2000]

# The collate_fn pads a batch of variable-length sequences to the longest
# one, one-hot encodes Y over the vocabulary, and returns (X, Y, mask).
train_loader = DataLoader(
    dataset,
    batch_size=16,
    shuffle=True,
    collate_fn=dataset.collate,
)

# --- Device auto-detect: CUDA > MPS > CPU ---
DEVICE = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

# --- Tiny-model hyperparameters ---
# Small enough to train in <1 minute on Colab CPU, large enough to learn
# something interesting from 2000 lines of Shakespeare.
CTX_LEN  = 128   # maximum sequence length the positional embedding supports
D_MODEL  = 64    # hidden dimension (embedding size)
N_LAYERS = 2     # number of stacked transformer blocks
N_HEADS  = 4     # number of attention heads per block (64 / 4 = 16 per head)

print(f"Vocab size:  {VOCAB_SIZE}")
print(f"Pad index:   {PAD_IDX}")
print(f"Dataset:     {len(dataset)} (X, Y) pairs")
print(f"Device:      {DEVICE}")
print("Setup complete.")


/content
Cloning into 'nd-cse-10124-lectures'...
remote: Enumerating objects: 466, done.
remote: Counting objects: 100% (55/55), done.
remote: Compressing objects: 100% (43/43), done.
remote: Total 466 (delta 19), reused 41 (delta 11), pack-reused 411 (from 1)
Receiving objects: 100% (466/466), 60.58 MiB | 25.29 MiB/s, done.
Resolving deltas: 100% (282/282), done.
/content/nd-cse-10124-lectures
/content/nd-cse-10124-lectures
Vocab size:  512
Pad index:   256
Dataset:     2000 (X, Y) pairs
Device:      cpu
Setup complete.


---

## Task 01: Lay Out the Architecture

### Why This Particular Stack?

A GPT language model is the minimum number of sub-modules that let you do **autoregressive next-token prediction** well. Each group in the stack has a clear job:

- **Token embedding** — the model needs real-valued vectors, not integers, so the first step is always a lookup table from token ID to vector. Entries are learned.
- **Positional embedding** — pure self-attention is *permutation-invariant*: it has no idea whether a token is first or last in the sequence. The positional embedding breaks that symmetry by adding a different learnable vector at each position.
- **Transformer blocks** — the part that actually learns language. Each block mixes information *between* tokens via attention and then processes each token independently via the feed-forward network. Stacking more blocks lets the model compose more complex features.
- **Final layer norm** — a stability trick. Without a final LN, the residual stream's magnitude can drift during training, making the LM head's logits hard to train. GPT-2 added this and it stuck.
- **Language model head** — a plain linear projection from $d_\text{model}$ to the vocabulary size $V$. It outputs one *logit* per word per position, which the softmax turns into a probability distribution.

### The Shapes

At every step the batch dimension $B$ and sequence length $T$ stay the same; only the last dimension changes.

| Step | Tensor | Shape |
|------|--------|-------|
| Input | $X$ | $(B, T)$ int64 |
| After token embedding | $\text{tok}$ | $(B, T, C)$ |
| After positional embedding | $\text{tok} + \text{pos}$ | $(B, T, C)$ |
| After each transformer block | $h$ | $(B, T, C)$ |
| After final layer norm | $h$ | $(B, T, C)$ |
| After LM head | $\text{logits}$ | $(B, T, V)$ |
| After softmax | $\text{probs}$ | $(B, T, V)$ |

where $C = $ `D_MODEL` and $V = $ `VOCAB_SIZE`.


### Task 01-1: Description (0 pts.)
#### Instantiate Sub-Modules

In the cell below, use the imports and hyperparameter constants from Task 00 to create one instance of each sub-module and store them in **module-level variables** with exactly these names:

- `token_embedding`
- `positional_embedding`
- `transformer_blocks` — a Python **list** of length `N_LAYERS` (not a single block)
- `final_ln`
- `lm_head`

Every later task will reference these by name, so the names matter.

**Constructor signatures** (all sub-modules take a `device=DEVICE` keyword):

| Sub-module | Constructor |
|---|---|
| `EmbeddingLayer` | `EmbeddingLayer(vocab_size, d_model, device=...)` |
| `PositionalEmbedding` | `PositionalEmbedding(ctx_len, d_model, device=...)` |
| `Transformer` (the block from Lab 03) | `Transformer(d_model, n_heads=..., use_gelu=True, device=..., vocab_size=...)` |
| `LayerNorm` | `LayerNorm(d_model, device=...)` |
| `LinearLayer` | `LinearLayer(d_model, vocab_size, device=...)` |

**Hints:**
- `transformer_blocks` is a list. The clean way to build it is a list comprehension: `[Transformer(...) for _ in range(N_LAYERS)]`.
- The `Transformer` constructor takes `vocab_size=VOCAB_SIZE` even though the block itself doesn't directly produce vocab-sized outputs; just pass it along.
- `lm_head`'s output dimension must equal `VOCAB_SIZE` — this is what lets the model produce one logit per vocabulary word.

### Task 01-1: Code (1 pt.)


In [26]:
# Step 1: Create the token embedding lookup table.
#   It maps each of the VOCAB_SIZE token IDs to a D_MODEL-dimensional vector.
#   Entries are learned from scratch.
# TODO: create token_embedding
# LINES: 1
token_embedding = EmbeddingLayer(VOCAB_SIZE, D_MODEL, device=DEVICE)

# Step 2: Create the positional embedding lookup table.
#   It stores one learnable D_MODEL vector for each of the CTX_LEN positions.
#   At forward time we will add it to the token embedding to give the model
#   "where am I in the sequence" information.
# TODO: create positional_embedding
# LINES: 1
positional_embedding = PositionalEmbedding(CTX_LEN, D_MODEL, device=DEVICE)

# Step 3: Create a list of N_LAYERS identical Transformer blocks.
#   Each block has its own independent weights (NOT shared across blocks).
#   A list comprehension is the cleanest way to build this.
# TODO: create transformer_blocks as a list of Transformer instances
# LINES: 3

transformer_blocks = [
    Transformer(D_MODEL, n_heads=N_HEADS, use_gelu=True, device=DEVICE)
    for _ in range(N_LAYERS)
]

# Step 4: Create the final LayerNorm applied right before the LM head.
#   This is the "ln_f" in GPT-2 terminology.
# TODO: create final_ln
# LINES: 1
final_ln = LayerNorm(D_MODEL, device=DEVICE)

# Step 5: Create the language-model head.
#   It is a plain LinearLayer that projects from D_MODEL down to VOCAB_SIZE,
#   producing one logit per vocabulary word per position.
# TODO: create lm_head
# LINES: 1
lm_head = LinearLayer(D_MODEL, VOCAB_SIZE, device=DEVICE)


In [27]:
print("Architecture laid out:")
print(f"  token_embedding:      {type(token_embedding).__name__}")
print(f"  positional_embedding: {type(positional_embedding).__name__}")
print(f"  transformer_blocks:   list of {len(transformer_blocks)} {type(transformer_blocks[0]).__name__}")
print(f"  final_ln:             {type(final_ln).__name__}")
print(f"  lm_head:              {type(lm_head).__name__}")

Architecture laid out:
  token_embedding:      EmbeddingLayer
  positional_embedding: PositionalEmbedding
  transformer_blocks:   list of 2 Transformer
  final_ln:             LayerNorm
  lm_head:              LinearLayer


### Task 01-1: Expected Output (0.5 pts.)

Your instantiation cell should not raise. Running a simple `type(...)` check like the solution prints should say:

```
Architecture laid out:
  token_embedding:      EmbeddingLayer
  positional_embedding: PositionalEmbedding
  transformer_blocks:   list of 2 Transformer
  final_ln:             LayerNorm
  lm_head:              LinearLayer
```


### Task 01-2: Description (0 pts.)
#### Parameter Count Helper

A good sanity check on any neural network is "how many parameters does it have?" A parameter is any scalar the model will update during training — every entry of every weight matrix, every bias vector.

Write a `count_parameters()` helper that sums `.numel()` across every tensor attribute of every sub-module you created in Task 01-1. For the tiny configuration (`D_MODEL=64`, `N_LAYERS=2`, `N_HEADS=4`, `VOCAB_SIZE=512`, `CTX_LEN=128`) you should get roughly **70k–80k parameters**.

**Approach:** our from-scratch sub-modules store their learnable tensors as attributes (e.g. `LinearLayer` has `.W` and `.b`; `EmbeddingLayer` has `.W`; `LayerNorm` has `.gamma` and `.beta`). You do *not* have to hard-code each attribute name — you can discover them at runtime by iterating over `dir(obj)` and filtering for `torch.Tensor` attributes. That keeps the helper robust if the sub-modules ever add new weights.

**Useful functions:**
- `dir(obj)` — returns a list of all attribute names on `obj`
- `getattr(obj, name, None)` — safely fetch an attribute by name, returning `None` if missing
- `isinstance(x, torch.Tensor)` — True if `x` is a PyTorch tensor
- `x.numel()` — total number of elements in a tensor

### Task 01-2: Code (1 pt.)


In [28]:
def count_parameters():
    # TODO: iterate over every sub-module (token_embedding, positional_embedding,
    #       each block in transformer_blocks, final_ln, lm_head) and sum
    #       .numel() across its torch.Tensor attributes. Return the total.
    # LINES: ~15
    total = 0

    modules = [
        token_embedding,
        positional_embedding,
        *transformer_blocks,
        final_ln,
        lm_head,
    ]

    for module in modules:
        for attr_name in dir(module):
            attr = getattr(module, attr_name, None)
            if isinstance(attr, torch.Tensor):
                total += attr.numel()



    return total

print(f"Tiny model has ~{count_parameters():,} parameters.")


Tiny model has ~74,368 parameters.


### Task 01-2: Expected Output (0.5 pts.)

```
Tiny model has ~74,368 parameters.
```

Don't panic if yours is off by a few hundred — some implementations of the `Transformer` block add optional biases; anything in the 70k–80k range is fine.


---

## Task 02: Write `forward`

### What `forward` Does

`forward(X)` is the model's **prediction** function. Given a batch of token IDs, it returns, for every position, a probability distribution over what the *next* token will be.

Mathematically, for one sequence of tokens $x_1, x_2, \dots, x_T$ the model computes:

$$P(x_{i+1} \mid x_1, \dots, x_i) = \text{softmax}\big(\text{logits}_i\big)$$

for every position $i$ in parallel. The transformer block's causal masking (which you built in Lab 03) is what guarantees that position $i$'s prediction can only depend on positions $1, \dots, i$, never the future.

### The Data Flow — Step by Step

```
X  (B, T)  int64 token IDs
│
├─► key_pad_mask = (X == PAD_IDX)         (B, T)  bool

│
├─► tok = token_embedding.forward(X)                 (B, T, C)
├─► pos = positional_embedding.forward(B, T)         (B, T, C)
│    h  = tok + pos                                  (B, T, C)
│
├─► for block in transformer_blocks:
│       h = block.forward(h, key_pad_mask)           (B, T, C)
│
├─► h      = final_ln.forward(h)                     (B, T, C)
├─► logits = lm_head.forward(h)                      (B, T, V)
└─► probs  = softmax(logits, dim=-1)                 (B, T, V)
```

### Things To Notice

- **Adding, not concatenating, the positional embedding.** Both `tok` and `pos` are shape $(B, T, C)$. Adding them gives each token both its identity *and* its position in the same vector — the model is free to use whichever it needs.
- **The `key_pad_mask`.** Because a batch contains sequences of different lengths, the dataset's `collate` function pads the short ones with `PAD_IDX`. We pass a boolean mask telling the attention layers which positions to ignore so padding doesn't poison the attention weights.
- **Softmax on the last dimension.** We want probabilities over the vocabulary *for each position independently*, so the softmax is along `dim=-1`. If you softmax along the wrong dimension the probabilities won't sum to 1 over the vocabulary.
- **`PositionalEmbedding.forward` takes two integers**, not a tensor: `positional_embedding.forward(batch_size=B, seq_len=T)`. It returns a $(B, T, C)$ tensor you can add directly to `tok`.


### Worked example — one tiny batch end-to-end

Imagine `VOCAB_SIZE=4`, `D_MODEL=2`, `B=1`, `T=3`. The tokenizer maps `"hi !"` to the token IDs `[1, 2, 3]`. Let's trace what every shape looks like:

```
X = [[1, 2, 3]]                   shape (1, 3)   int64

key_pad_mask = (X == PAD_IDX)     shape (1, 3)   bool
              = [[False, False, False]]   (no padding in this tiny example)

tok = token_embedding.forward(X)        shape (1, 3, 2)
    = [[[ 0.4,  0.1],
        [-0.2,  0.5],
        [ 0.3, -0.3]]]

pos = positional_embedding.forward(B=1, T=3)   shape (1, 3, 2)
    = [[[ 0.0,  0.0],
        [ 0.1,  0.0],
        [ 0.2,  0.0]]]

h = tok + pos                            shape (1, 3, 2)
  = [[[ 0.4,  0.1],
      [-0.1,  0.5],
      [ 0.5, -0.3]]]

# ... after passing through the transformer blocks and final_ln, h is still (1, 3, 2) ...

logits = lm_head.forward(h)              shape (1, 3, 4)   one row per position, one column per vocab word
       = [[[ 1.2, -0.5,  0.3, -0.1],
          [ 0.8,  0.4, -0.2,  0.0],
          [-0.1,  0.0,  1.5,  0.2]]]

probs = softmax(logits, dim=-1)          shape (1, 3, 4)   each row sums to 1
      = [[[0.55, 0.10, 0.22, 0.13],
          [0.40, 0.27, 0.15, 0.18],
          [0.10, 0.11, 0.51, 0.28]]]
```

Notice three things: (1) the batch and sequence dimensions never change shape — only the last dimension transforms; (2) `probs` at position `t` is the model's distribution over what the **next** token will be; (3) every row in `probs` sums to 1 because softmax normalizes along the last axis.

### Task 02-1: Description (0 pts.)
#### `forward(X)` Function

Implement the data flow pictured above. You will need to fill in the body of `forward`. The function has five logical steps: compute the `key_pad_mask`, add token + positional embeddings, run the transformer block stack, apply the final LN, and project + softmax to get probabilities.

**Useful functions:**
- `token_embedding.forward(X)` — takes `(B, T)` int64, returns `(B, T, C)` float
- `positional_embedding.forward(batch_size=B, seq_len=T)` — returns `(B, T, C)` float
- `block.forward(h, key_pad_mask=key_pad_mask)` — takes `(B, T, C)`, returns `(B, T, C)`
- `final_ln.forward(h)` — takes `(B, T, C)`, returns `(B, T, C)`
- `lm_head.forward(h)` — takes `(B, T, C)`, returns `(B, T, V)`
- `F.softmax(x, dim=-1)` — softmax along the last dimension (already imported as `F`)

### Task 02-1: Code (1 pt.)


In [29]:
def forward(X):
    # Full forward pass of the tiny GPT language model.
    #   X: (B, T) int64 token IDs on DEVICE.
    # Returns:
    #   (B, T, V) probabilities that sum to 1 along the last dim.
    B, T = X.shape

    # Step 1: Build a boolean mask marking which positions in X are padding.
    #   This gets passed to each transformer block so attention ignores
    #   padded positions. Shape: (B, T) bool.
    # TODO: compute key_pad_mask
    # LINES: 1
    key_pad_mask = (X == PAD_IDX)

    # Step 2: Look up the token and positional embeddings and add them.
    #   token embedding says "which word is this?"; positional embedding
    #   says "where in the sequence is it?". Element-wise addition combines
    #   both into a single (B, T, C) representation.
    # TODO: call token_embedding.forward and positional_embedding.forward,
    #       then add them into a variable named h.
    # LINES: 3
    tok = token_embedding.forward(X)
    pos = positional_embedding.forward(batch_size=B, seq_len=T)
    h = tok + pos

    # Step 3: Run h through every transformer block in sequence.
    #   Each block refines the representation by mixing information across
    #   tokens (attention) and processing each token independently (FFN).
    # TODO: loop over transformer_blocks and update h
    # LINES: 2
    for block in transformer_blocks:
        h = block.forward(h, key_pad_mask=key_pad_mask)

        pass

    # Step 4: Final layer normalization before the LM head.
    # TODO: apply final_ln
    # LINES: 1
    h = final_ln.forward(h)

    # Step 5: Project to vocabulary logits and softmax into probabilities.
    # TODO: call lm_head.forward, then softmax over the last dimension
    # LINES: 2
    logits = lm_head.forward(h)
    probs = F.softmax(logits, dim=-1)

    return probs


### Task 02-2: Sanity Check (0.5 pts.)

Pull one batch from `train_loader` and call your `forward`. Verify two things:

1. The output shape is exactly `(B, T, VOCAB_SIZE)`.
2. The output is a valid probability distribution at every position — i.e. for every `(b, t)`, the probabilities over the vocabulary sum to 1 (within floating-point tolerance).

If either check fails, go back and debug `forward` before moving on — the rest of the lab depends on it.


In [30]:
# Step 1: Pull one batch from the train_loader.
#   collate returns a 3-tuple: (X, Y, mask). We won't use mask in this cell.
# TODO: get one batch from the loader
# LINES: 1
X_batch, Y_batch, mask_batch = next(iter(train_loader))

# Step 2: Run forward to get the predicted probabilities.
# TODO: call forward(X_batch)
# LINES: 1
probs = forward(X_batch)

# Step 3: Verify the output shape is exactly (B, T, VOCAB_SIZE).
#   Print probs.shape and assert it matches.
# TODO: print + assert
# LINES: 2

B, T = X_batch.shape
print(f"probs.shape = {probs.shape}")
assert probs.shape == (B, T, VOCAB_SIZE), f"Unexpected shape: {probs.shape}"

# Step 4: Verify probs sum to 1 along the vocab dimension at every (b, t).
#   Use torch.allclose(row_sums, ones, atol=1e-4) so small floating-point
#   drift doesn't trip the check.
# TODO: compute row_sums and assert torch.allclose(...)
# LINES: 2

row_sums = probs.sum(dim=-1)
assert torch.allclose(row_sums, torch.ones_like(row_sums), atol=1e-4)


print("Sanity check passed: probabilities sum to 1 at every position.")


probs.shape = torch.Size([16, 24, 512])
Sanity check passed: probabilities sum to 1 at every position.


---

## Task 03: Write `backward`

### Why We Need `backward`

Training any neural network is just gradient descent on its loss. To do a gradient descent step you need the **gradient of the loss with respect to every parameter** — i.e. "if I wiggle this weight up a tiny bit, does the loss go up or down, and by how much?"

Our sub-modules each already know how to compute their local gradients: every class you wrote (or imported) has a `.backward(dA)` method that takes the gradient of the loss with respect to *that layer's output* and returns the gradient with respect to its *input* (while also accumulating gradients for its own parameters). All the top-level `backward` has to do is:

1. Compute the **initial gradient** at the very top of the graph.
2. Hand it to the sub-modules in **reverse order**.

That's it. This is what "backpropagation" means in practice.

### The Initial Gradient

Our loss is masked cross-entropy over a softmax output:

$$\mathcal{L} = -\frac{1}{N} \sum_{(b,t) \in \text{real}} \sum_{v=1}^{V} Y_{b,t,v} \log \hat{Y}_{b,t,v}$$

where $\hat Y = \text{softmax}(\text{logits})$, $Y$ is one-hot, and $N$ is the number of real (non-padded) tokens.

If you grind through the chain rule for **softmax + cross-entropy together** (the magic cancellation Prof. Karpathy keeps raving about), the gradient with respect to the logits simplifies to:

$$\frac{\partial \mathcal{L}}{\partial \text{logits}_{b,t,v}} = \frac{(\hat Y_{b,t,v} - Y_{b,t,v}) \cdot \text{mask}_{b,t}}{N}$$

In code:

```python
mask       = (Y.argmax(dim=-1) != PAD_IDX).unsqueeze(-1).float()   # (B, T, 1)
normalizer = mask.sum().clamp_min(1.0)                             # scalar
dA         = (Y_hat - Y) * mask / normalizer                       # (B, T, V)
```

`mask` zeros out the gradient at padded positions (we don't want the model to learn to predict padding), and `normalizer` divides by the number of real tokens so the gradient magnitude doesn't depend on batch size or padding.

### The Reverse-Order Chain

Once you have `dA` at the logits, you hand it to the sub-modules in **exactly the reverse of the forward order**:

```
forward:  token_embedding -> positional_embedding -> blocks -> final_ln -> lm_head
backward:  lm_head -> final_ln -> blocks (reversed) -> positional_embedding + token_embedding
```

Each `.backward` call returns the gradient with respect to its input, which you then pass to the next layer down. The very bottom layers (the embeddings) don't pass anything further down — they just stash their own parameter gradients and return.


### Worked example — what the initial gradient actually looks like

Take a single position with `VOCAB_SIZE=3`. Suppose the model predicted:

```
Y_hat = [0.7, 0.2, 0.1]    (model is 70% sure the answer is class 0)
Y     = [0.0, 1.0, 0.0]    (true answer is actually class 1; this is one-hot)
```

The closed-form `(softmax + cross-entropy)'` gradient at the logits is just `Y_hat - Y` (times the mask and divided by `N`, but for a single non-padded token both are 1):

```
dA = Y_hat - Y = [0.7, 0.2, 0.1] - [0.0, 1.0, 0.0]
              = [+0.7, -0.8, +0.1]
```

Read the signs:
- `+0.7` on class 0 → "you over-confidently said this; **push the logit DOWN**."
- `-0.8` on class 1 → "you under-shot the right answer; **push the logit UP**." (gradient descent moves opposite the gradient sign.)
- `+0.1` on class 2 → small over-prediction; small downward push.

The mask multiplier turns this off at padded positions; the divisor `N` averages over all real tokens in the batch. Both are bookkeeping. The actual *learning signal* is `Y_hat - Y` — the difference between what the model thinks and what the truth is.

### Task 03-1: Description (0 pts.)
#### `backward(Y_hat, Y)` Function

Implement `backward`. It should take the predicted probabilities `Y_hat` from your `forward` and the one-hot targets `Y` from the DataLoader, compute the initial gradient, and pass it through the sub-modules in reverse.

**Useful functions:**
- `Y.argmax(dim=-1)` — takes a one-hot `(B, T, V)` tensor and returns the class index at each position `(B, T)`
- `mask.sum().clamp_min(1.0)` — total number of real tokens in the batch (at least 1, so we never divide by zero)
- `lm_head.backward(dA)` — returns the gradient with respect to `lm_head`'s input
- `final_ln.backward(dA)`, `block.backward(dA)`, `positional_embedding.backward(dA)`, `token_embedding.backward(dA)` — likewise
- `reversed(transformer_blocks)` — iterate over the block list back-to-front

**Order matters.** If you pass gradients in the wrong order you will get runtime errors (shape mismatch) or silently-wrong training (loss won't drop). Double-check the reverse chain before running.

### Task 03-1: Code (1 pt.)


In [31]:
def backward(Y_hat, Y):
    # Backward pass: propagate the cross-entropy gradient through every
    # sub-module in reverse order.
    #   Y_hat: (B, T, V) predicted probabilities from forward()
    #   Y:     (B, T, V) one-hot targets from the DataLoader

    # Step 1: Build the padding mask over target positions.
    #   We compare Y.argmax back to PAD_IDX because Y is one-hot.
    #   Shape: (B, T, 1) float so it broadcasts over the vocab dimension.
    # TODO: compute mask
    # LINES: 1
    mask = (Y.argmax(dim=-1) != PAD_IDX).unsqueeze(-1).float()

    # Step 2: Normalizer = number of real (non-padded) target tokens.
    #   clamp_min(1.0) protects against all-padding batches.
    # TODO: compute normalizer
    # LINES: 1
    normalizer = mask.sum().clamp_min(1.0)

    # Step 3: Compute the initial gradient at the logits.
    #   For softmax + cross-entropy, dL/d(logits) = (Y_hat - Y) * mask / N.
    # TODO: compute dA
    # LINES: 1
    dA = (Y_hat - Y) * mask / normalizer

    # Step 4: Backprop through the top layers first.
    #   Forward order was: ... -> final_ln -> lm_head, so backward is
    #   lm_head -> final_ln.
    # TODO: backward through lm_head and final_ln
    # LINES: 2
    # dA = lm_head.backward(...)
    # dA = final_ln.backward(...)

    dA = lm_head.backward(dA)
    dA = final_ln.backward(dA)

    # Step 5: Backprop through the transformer block stack in REVERSE order.
    #   Forward was block_0, block_1, ..., block_{N-1}; backward is
    #   block_{N-1}, ..., block_1, block_0.
    # TODO: loop over reversed(transformer_blocks)
    # LINES: 2
    for block in reversed(transformer_blocks):
        dA = block.backward(dA)

    # Step 6: The embeddings are the bottom of the graph. They absorb the
    #   incoming gradient into their own parameter gradients and don't
    #   propagate anything further down.
    # TODO: backward through positional_embedding and token_embedding
    # LINES: 2
    # positional_embedding.backward(...)
    # token_embedding.backward(...)

    positional_embedding.backward(dA)
    token_embedding.backward(dA)


### Task 03-2: Sanity Check (0.5 pts.)

The most common bug in a from-scratch backward pass is silently *not updating* something — you run the training loop, the loss stays flat, and you have no idea why. A cheap early-warning check is to run `backward` once on a single batch and verify that at least one of the sub-module parameter gradients is actually non-zero.

`lm_head.dW` is a great candidate because it's the very top of the backward chain: if `dA` made it there at all, `lm_head.dW` will have non-zero entries.


In [32]:
# Step 1: Run backward on the (probs, Y_batch) you got in Task 02-2.
#   This walks the gradient through every sub-module and stores the
#   parameter gradients on each one (e.g. lm_head.dW, final_ln.dgamma, ...).
# TODO: call backward(probs, Y_batch)
# LINES: 1
backward(probs, Y_batch)

# Step 2: Verify lm_head.dW received a non-zero gradient.
#   If grad_norm is exactly 0.0, the gradient never made it down the
#   backward chain (most likely a return-value or order bug in backward).
#   Print it so you can eyeball its scale.
# TODO: compute grad_norm = lm_head.dW.abs().sum().item()
# TODO: assert grad_norm > 0.0
# TODO: print grad_norm
# LINES: ~3

grad_norm = lm_head.dW.abs().sum().item()
assert grad_norm > 0.0
print(f"lm_head.dW grad_norm = {grad_norm}")

lm_head.dW grad_norm = 66.59146118164062


---

## Task 04: Training Loop

### The Four Steps, Repeated

Every training loop for every neural network in history is structurally the same four steps, run over and over on batch after batch of data:

1. **Forward pass:** $\hat Y = \text{model}(X)$
2. **Loss:** $\mathcal{L} = \text{cross\_entropy}(\hat Y, Y)$
3. **Backward pass:** compute $\partial \mathcal{L} / \partial \theta$ for every parameter $\theta$
4. **Update:** $\theta \leftarrow \theta - \text{lr} \cdot \partial \mathcal{L}/\partial \theta$

That's it. Bigger models and fancier optimizers change details of step 4 (momentum, adaptive learning rates, weight decay) but the shape of the loop is always these four steps.

You already have the forward and backward passes from Tasks 02 and 03. In this task you will:

1. Write the masked `cross_entropy` loss (step 2).
2. Bundle steps 1–4 into a single `step(X, Y, lr)` helper.
3. Wrap a few epochs of `train_loader` around `step` and watch the loss go down.

### Cross-Entropy in One Equation

For a single token position with one-hot target $Y$ and predicted distribution $\hat Y$:

$$\mathcal{L} = -\sum_{v=1}^{V} Y_v \log \hat Y_v = -\log \hat Y_{c}$$

where $c$ is the correct class. In plain English: **"how much probability did you assign to the right answer?"**  Lower loss means the model was more confident about the right token.

For a batch, we average this over every real (non-padded) position:

$$\mathcal{L}_{\text{batch}} = -\frac{1}{N} \sum_{(b,t) \in \text{real}} \log \hat Y_{b,t,c_{b,t}}$$

### What To Expect From the Loss Curve

The **untrained** model outputs a nearly uniform distribution over the vocabulary. Under a uniform distribution the expected cross-entropy is exactly:

$$\mathcal{L}_{\text{uniform}} = -\log \frac{1}{V} = \log V$$

For `VOCAB_SIZE = 512`, that's $\log 512 \approx 6.24$. You should see the *first* loss measurement around **6.2-7.5** (a bit higher than log V because of random initialization noise), then steadily dropping toward something in the **5-6** range after 5 epochs. If your loss stays stuck near $\log V$ the backward pass isn't working; if it jumps around wildly the learning rate is too high.


### Worked example — cross-entropy with numbers

Same `VOCAB_SIZE=3` setup. Two scenarios for a single token where the true class is `1`:

**Scenario A — model is right and confident:**
```
Y_hat = [0.1, 0.8, 0.1]      probability assigned to the right answer = 0.8
loss  = -log(0.8) ≈ 0.223
```

**Scenario B — model is wrong and confident:**
```
Y_hat = [0.8, 0.1, 0.1]      probability assigned to the right answer = 0.1
loss  = -log(0.1) ≈ 2.303
```

The wrong-and-confident answer gets ~10× the loss of the right-and-confident answer. That is exactly why cross-entropy works as a training signal: confident-and-correct ≈ 0 loss, confident-and-wrong is heavily penalized.

For a freshly-initialized model whose `softmax` output is roughly uniform across `V = 512` words, every position assigns about `1/512 ≈ 0.002` probability to the correct token, giving:

```
loss ≈ -log(1/512) = log(512) ≈ 6.24
```

That is your "untrained baseline" \u2014 your training loop should start near 6.24 and trend down from there.

### Task 04-1: Description (0 pts.)
#### Masked `cross_entropy`

Implement masked cross-entropy exactly as in the formula above, with two important numerical details:

1. **Clamp before logging.** `torch.log(0) = -inf`, which poisons any downstream arithmetic. Clamp `Y_hat` to be at least `1e-12` before taking the log.
2. **Mask padding.** We do not want the loss (or the gradient) to depend on padded positions. Build a `(B, T)` boolean mask `Y.argmax(dim=-1) != PAD_IDX` and multiply the per-token loss by it.

**Useful functions:**
- `Y_hat.clamp_min(1e-12)` — element-wise max with `1e-12`
- `torch.log(x)` — element-wise natural log
- `.sum(dim=-1)` / `.sum()` — summing over dimensions
- `mask.sum().clamp_min(1.0)` — number of real tokens, floored at 1 so we never divide by zero

### Task 04-1: Code (0.5 pts.)


In [33]:
def cross_entropy(Y_hat, Y):
    # Masked cross-entropy loss, averaged over real (non-padded) tokens.
    #   Y_hat: (B, T, V) predicted probabilities
    #   Y:     (B, T, V) one-hot targets

    # Step 1: Clamp Y_hat to avoid log(0), then take the log.
    # TODO: compute log_probs
    # LINES: 1
    log_probs = torch.log(Y_hat.clamp_min(1e-12))

    # Step 2: Per-token cross-entropy = -sum over vocab of Y * log Y_hat.
    #   Because Y is one-hot, this picks out -log(Y_hat[correct_class]) at
    #   each position. Shape after the sum: (B, T).
    # TODO: compute token_loss
    # LINES: 1
    token_loss = -torch.sum(Y * log_probs, dim=-1)

    # Step 3: Padding mask over target positions, shape (B, T) float.
    # TODO: compute mask
    # LINES: 1
    mask =  (Y.argmax(dim=-1) != PAD_IDX).float()

    # Step 4: Divide by the number of real tokens (clamped at 1 to avoid
    #   divide-by-zero on an all-padding batch).
    # TODO: compute normalizer
    # LINES: 1
    normalizer = mask.sum().clamp_min(1.0)

    # Step 5: Masked average.
    # TODO: return the masked-and-averaged loss
    # LINES: 1
    return (token_loss * mask).sum() / normalizer


### Task 04-1: Expected Output (0.5 pts.)

An untrained model on one batch should give you a loss near $\log 512 \approx 6.24$:

```python
untrained_loss = cross_entropy(forward(X_batch), Y_batch).item()
print(f"untrained loss: {untrained_loss:.4f}   log(V): {math.log(VOCAB_SIZE):.4f}")
```

```
untrained loss: 7.2029   log(V): 6.2383
```

Don't sweat the exact number — anything between 6 and 8 is fine. The point is that it should be *near* $\log V$, not orders of magnitude off.


### Task 04-2: Description (0 pts.)
#### `step` Function

`step(X, Y, lr)` is the entire "train on one batch" operation. Wire together the functions you just wrote into one call that does all four training steps in order:

1. `Y_hat = forward(X)`
2. `loss  = cross_entropy(Y_hat, Y)` — remember to `.item()` it before returning so we store a Python float, not a tensor
3. `backward(Y_hat, Y)`
4. Call `.update(lr)` on every sub-module: `token_embedding`, `positional_embedding`, each block in `transformer_blocks`, `final_ln`, `lm_head`

Then return the loss as a Python float.

**Why the loop-over-sub-modules for the update?** Our from-scratch framework doesn't have a PyTorch-style `optimizer.step()` that knows about every parameter — each sub-module owns its own `.update(lr)` method that applies `W -= lr * dW` to its own weights. The top-level `step` has to hand the learning rate to each one.

### Task 04-2: Code (1 pt.)


In [34]:
def step(X, Y, lr):
    # Run one full training step on one batch.
    # Returns the scalar loss value as a Python float.

    # Step 1: Forward pass.
    # TODO: compute Y_hat
    # LINES: 1
    Y_hat = forward(X)

    # Step 2: Compute the loss. Convert to Python float with .item().
    # TODO: compute loss
    # LINES: 1
    loss = cross_entropy(Y_hat, Y).item()

    # Step 3: Backward pass.
    # TODO: call backward
    # LINES: 1

    backward(Y_hat, Y)

    # Step 4: Update every sub-module's parameters.
    # TODO: call .update(lr) on token_embedding, positional_embedding,
    #       every transformer block, final_ln, and lm_head.
    token_embedding.update(lr)
    positional_embedding.update(lr)
    for block in transformer_blocks:
        block.update(lr)
    final_ln.update(lr)
    lm_head.update(lr)


    # LINES: 5

    return loss


### Task 04-3: Description (0 pts.)
#### Run the Training Loop

Finally, the payoff: actually train the model.

Before the loop, measure `initial_loss` once on a single batch using `forward` + `cross_entropy` (without calling `backward` or `update`). This is your "untrained baseline" — you will compare the final epoch loss to this value.

Then loop for `EPOCHS = 5` with `LR = 1e-3`. In each epoch:
1. Walk over every batch yielded by `train_loader` (remember that `collate` returns `(X, Y, mask)`, so you unpack three values).
2. Call `step(X, Y, LR)` and collect the returned loss into a list for the epoch.
3. Take the mean of that list and append it to `loss_history`.
4. Print the epoch number and average loss so you can watch it go down live.

After the loop, assert that your final epoch's loss is at least 10% lower than `initial_loss`. On the tiny model with 5 epochs on 2000 Shakespeare lines you should see the loss drop from around 7 down to around 6 — a ~14% drop. If your assert fails, go back and check your forward/backward/update.

### Task 04-3: Code (0.5 pts.)


In [35]:
import numpy as np
EPOCHS = 10
LR = 1e-3

# Step 1: Measure the untrained baseline on one batch.
# TODO: pull one batch, run forward + cross_entropy, store as initial_loss
# LINES: ~3
initial_loss = 0.0
X, Y, mask = next(iter(train_loader))
initial_loss = cross_entropy(forward(X), Y).item()


# Step 2: Train.
loss_history = []
# TODO: loop EPOCHS times over train_loader, call step(), record per-epoch
#       average losses into loss_history, and print progress.
# LINES: ~8
for epoch in range(EPOCHS):
    loss_history_epoch = []
    for X, Y, mask in train_loader:
        loss = step(X, Y, LR)

        loss_history_epoch.append(loss)
    loss_history.append(np.mean(loss_history_epoch))
    print(f"Epoch {epoch + 1}/{EPOCHS} - avg loss: {loss_history[-1]:.4f}")


# Step 3: Sanity assert — the training loop actually worked.
# TODO: assert loss_history[-1] < 0.9 * initial_loss
# LINES: ~3
assert loss_history[-1] < 0.9 * initial_loss

print(f"Initial loss: {initial_loss:.4f}")
print(f"Target:       {0.9 * initial_loss:.4f}")
print(f"Final:        {loss_history[-1]:.4f}")


Epoch 1/10 - avg loss: 6.9552
Epoch 2/10 - avg loss: 6.6194
Epoch 3/10 - avg loss: 6.4265
Epoch 4/10 - avg loss: 6.3000
Epoch 5/10 - avg loss: 6.2073
Epoch 6/10 - avg loss: 6.1325
Epoch 7/10 - avg loss: 6.0710
Epoch 8/10 - avg loss: 6.0180
Epoch 9/10 - avg loss: 5.9719
Epoch 10/10 - avg loss: 5.9335
Initial loss: 7.1949
Target:       6.4754
Final:        5.9335


### Task 04-3: Expected Output (0.5 pts.)

Your exact numbers will differ (random init, stochastic batch order), but the trajectory should look roughly like:

```
Initial loss (untrained): 7.2029
Epoch 1/5 - avg loss: 6.9720
Epoch 2/5 - avg loss: 6.5882
Epoch 3/5 - avg loss: 6.3829
Epoch 4/5 - avg loss: 6.2670
Epoch 5/5 - avg loss: 6.1889
Loss dropped as expected.
```

The key property: a strictly decreasing sequence with the final value at least 10% below the initial one. If your loss *increases*, your forward/backward/update is wrong — most likely an update-step sign error or a sub-module that isn't getting its `.update(lr)` called.


---

## Task 05: Generate Submission (0 pts.)

Run the cell below to write out `submission.json`. Replace `"YOUR_ID"` with your netID before running it, then upload both this notebook and `submission.json` to Gradescope.


In [36]:
import json

submission = {
    "lab": 4,
    "student_id": "atomashp",
    "num_parameters": int(count_parameters()),
    "initial_loss": float(initial_loss),
    "loss_history": [float(x) for x in loss_history],
}

with open("submission.json", "w") as f:
    json.dump(submission, f, indent=2)

print("Wrote submission.json")


Wrote submission.json


In [37]:
from google.colab import files
files.download("submission.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [38]:
import os, json

def export_notebook():
    L_PATH = "nd-cse-10124-lectures/Notebooks"
    L = "Lab_04_GPT2.0"

    try:
        from google.colab import _message, files

        repo_ipynb_path = f"/content/{L_PATH}/{L}.ipynb"
        nb = _message.blocking_request("get_ipynb", timeout_sec=1)["ipynb"]

        os.makedirs(os.path.dirname(repo_ipynb_path), exist_ok=True)
        with open(repo_ipynb_path, "w", encoding="utf-8") as f:
            json.dump(nb, f)

        !jupyter nbconvert --to html "{repo_ipynb_path}"
        files.download(repo_ipynb_path.replace(".ipynb", ".html"))
    except:
        import subprocess
        nb_fp = os.getcwd() + f'/{L}.ipynb'
        subprocess.run(["jupyter", "nbconvert", "--to", "html", nb_fp], check=True)

# TODO: Uncomment the line below and run to generate the file to submit on canvas
export_notebook()

[NbConvertApp] Converting notebook /content/nd-cse-10124-lectures/Notebooks/Lab_04_GPT2.0.ipynb to html
[NbConvertApp] Writing 395666 bytes to /content/nd-cse-10124-lectures/Notebooks/Lab_04_GPT2.0.html


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>